# Smooth DRLB tune with `train-derived lambda_init`

This notebook isolates a new smooth-DRLB experiment namespace with `inference_lambda_init_mode="train_derived"`.

Flow:

1. Build a clean `train/val` split inside this experiment folder.
2. Run a baseline manual configuration on `train/val`.
3. Tune with Optuna for `10` trials on `train/val`.
4. Re-train the best params on the original BAT `train` split.
5. Evaluate on the original BAT `test` split in the same style as the baseline-comparison notebooks.

All artifacts are saved under this experiment folder only.


In [1]:
import json
import sys
from pathlib import Path

import optuna
import pandas as pd
from IPython.display import display

notebook_dir = Path().resolve()
example_notebooks_dir = notebook_dir.parent.parent
bat_autobidding_dir = example_notebooks_dir.parent

if str(example_notebooks_dir) not in sys.path:
    sys.path.insert(0, str(example_notebooks_dir))
if str(bat_autobidding_dir) not in sys.path:
    sys.path.insert(0, str(bat_autobidding_dir))

from experiments.exp_configs import (
    RND42N10Config,
    RND42TrainTestHybridSmoothConfig,
    RND42TrainValHybridSmoothLambdaTrainConfig,
)
from simulator.model.drlb_bidder import DRLBBidder
from simulator.validation.check_results import autobidder_check

pd.set_option("display.max_columns", 200)


/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
source_config = RND42N10Config()
trainval_config = RND42TrainValHybridSmoothLambdaTrainConfig()
trainval_config.ensure_artifact_dirs()
trainval_config.config_dir.mkdir(parents=True, exist_ok=True)

traintest_source_config = RND42TrainTestHybridSmoothConfig()

OBJECTIVE = "clicks"
EXP_TYPE = "improved_hybrid_drlb_smooth_eval"
INFERENCE_LAMBDA_INIT_MODE = "train_derived"
VAL_FRACTION = 0.2
N_TRIALS = 10
MAX_TRAIN_STEPS = None  # Set for smoke runs.
VERBOSE = False

BASE_DRLB_PARAMS = {
    "max_bid": 100.0,
    "T": 72,
    "lambda_min": 1e-6,
    "lambda_max": 10.0,
    "bids_per_timestep": 1,
    "dqn_soft_update_tau": 0.01,
    "dqn_loss_type": "smooth_l1",
    "dqn_grad_clip_norm": 5.0,
    "dqn_reward_clip_value": 10.0,
    "inference_lambda_init_mode": INFERENCE_LAMBDA_INIT_MODE,
}

BASELINE_MODEL_PARAMS = {
    "dqn_gamma": 1.0,
    "dqn_lr": 1e-4,
    "dqn_target_update_interval": 100,
    "reward_net_lr": 1e-3,
}

print("Experiment dir:", trainval_config.experiment_dir)
print("Outputs dir:", trainval_config.outputs_dir)
print("Best models dir:", trainval_config.best_models_dir)
print("Best params dir:", trainval_config.best_params_dir)
print("Inference lambda mode:", INFERENCE_LAMBDA_INIT_MODE)


Experiment dir: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/exp_tune_smooth_drlb_dqn_lambda_train
Outputs dir: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/exp_tune_smooth_drlb_dqn_lambda_train/outputs
Best models dir: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/exp_tune_smooth_drlb_dqn_lambda_train/best_models
Best params dir: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/exp_tune_smooth_drlb_dqn_lambda_train/best_params
Inference lambda mode: train_derived


In [3]:
def ensure_train_val_split(source_config, target_config, val_fraction=0.2, seed=42):
    source_campaigns = pd.read_csv(source_config.data_config["train"]["campaigns_path"])
    source_stats = pd.read_csv(source_config.data_config["train"]["stats_path"])

    n_val = max(1, int(round(len(source_campaigns) * val_fraction)))
    val_campaigns = (
        source_campaigns
        .sample(n=n_val, random_state=seed)
        .sort_values("campaign_id")
        .reset_index(drop=True)
    )
    val_campaign_ids = set(val_campaigns["campaign_id"].astype(int).tolist())

    train_campaigns = (
        source_campaigns[~source_campaigns["campaign_id"].astype(int).isin(val_campaign_ids)]
        .sort_values("campaign_id")
        .reset_index(drop=True)
    )
    train_stats = source_stats[source_stats["campaign_id"].astype(int).isin(set(train_campaigns["campaign_id"].astype(int)))].copy()
    val_stats = source_stats[source_stats["campaign_id"].astype(int).isin(val_campaign_ids)].copy()

    train_campaigns.to_csv(target_config.data_config["train"]["campaigns_path"], index=False)
    train_stats.to_csv(target_config.data_config["train"]["stats_path"], index=False)
    val_campaigns.to_csv(target_config.data_config["test"]["campaigns_path"], index=False)
    val_stats.to_csv(target_config.data_config["test"]["stats_path"], index=False)

    metadata = {
        "source_experiment": source_config.experiment_name,
        "target_experiment": target_config.experiment_name,
        "seed": int(seed),
        "val_fraction": float(val_fraction),
        "train_campaigns": int(len(train_campaigns)),
        "val_campaigns": int(len(val_campaigns)),
        "train_stats_rows": int(len(train_stats)),
        "val_stats_rows": int(len(val_stats)),
    }
    metadata_path = target_config.config_dir / "train_val_split_metadata.json"
    metadata_path.write_text(json.dumps(metadata, indent=2))
    return metadata


def score_to_dict(score, skipped_campaigns, time_inference_sec, time_overall_sec):
    return {
        "cpc_relative": float(score[0]),
        "rmse": float(score[1]),
        "clicks_sum": float(score[2]),
        "quickspend": float(score[3]),
        "skipped_campaigns": int(skipped_campaigns),
        "time_inference_sec": float(time_inference_sec),
        "time_overall_sec": float(time_overall_sec),
    }


def summarize_diagnostics(diagnostics_df):
    if diagnostics_df.empty:
        return {
            "train_steps": 0,
            "last_dqn_loss": None,
            "last_reward_net_loss": None,
            "dqn_loss_mean": None,
            "dqn_loss_p95": None,
            "reward_net_loss_mean": None,
            "reward_net_loss_p95": None,
            "reward_signal_mean": None,
            "lambda_final": None,
        }

    dqn_loss_nonzero = diagnostics_df.loc[diagnostics_df["dqn_loss"] > 0, "dqn_loss"]
    reward_net_nonzero = diagnostics_df.loc[diagnostics_df["reward_net_loss"] > 0, "reward_net_loss"]
    return {
        "train_steps": int(len(diagnostics_df)),
        "last_dqn_loss": float(diagnostics_df["dqn_loss"].iloc[-1]),
        "last_reward_net_loss": float(diagnostics_df["reward_net_loss"].iloc[-1]),
        "dqn_loss_mean": float(dqn_loss_nonzero.mean()) if not dqn_loss_nonzero.empty else None,
        "dqn_loss_p95": float(dqn_loss_nonzero.quantile(0.95)) if not dqn_loss_nonzero.empty else None,
        "reward_net_loss_mean": float(reward_net_nonzero.mean()) if not reward_net_nonzero.empty else None,
        "reward_net_loss_p95": float(reward_net_nonzero.quantile(0.95)) if not reward_net_nonzero.empty else None,
        "reward_signal_mean": float(diagnostics_df["reward_signal"].mean()),
        "lambda_final": float(diagnostics_df["lambda"].iloc[-1]),
    }


def build_bidder_params(model_params, model_path=None, verbose=False):
    return {
        **BASE_DRLB_PARAMS,
        **model_params,
        "model_path": None if model_path is None else str(model_path),
        "exp_type": EXP_TYPE,
        "objective": OBJECTIVE,
        "eval_mode": True,
        "verbose": verbose,
        "use_tqdm": verbose,
        "debug_logs": False,
        "fit_log_every": 500,
        "inference_log_every": 24,
    }


def run_drlb_candidate(*, train_stats_df, train_campaigns_df, eval_campaigns_path, eval_stats_path, label, model_params, max_train_steps=None, verbose=False):
    bidder_params = build_bidder_params(model_params, model_path=None, verbose=verbose)
    bidder = DRLBBidder(bidder_params)
    bidder.fit(
        train_stats_df,
        campaigns_df=train_campaigns_df,
        max_steps=max_train_steps,
        objective=OBJECTIVE,
    )

    diagnostics = bidder.get_training_diagnostics().copy()
    diagnostics_path = trainval_config.outputs_dir / f"{label}_training_diagnostics.csv"
    diagnostics.to_csv(diagnostics_path, index=False)

    model_path = trainval_config.best_models_dir / f"{label}.pt"
    bidder.save_model(str(model_path))

    eval_params = {
        **build_bidder_params(model_params, model_path=model_path, verbose=verbose),
        "input_campaigns": eval_campaigns_path,
        "input_stats": eval_stats_path,
        "eval_mode": True,
    }
    result = autobidder_check(
        bidder=DRLBBidder,
        params=eval_params,
        auction_mode=trainval_config.auction_mode,
        verbose=verbose,
        log_every_campaigns=100,
        use_tqdm=verbose,
    )

    metrics = score_to_dict(
        score=result["score"],
        skipped_campaigns=result["skipped_campaigns"],
        time_inference_sec=result["time_inference_sec"],
        time_overall_sec=result["time_overall_sec"],
    )
    metrics.update({
        "label": label,
        "exp_type": EXP_TYPE,
        "inference_lambda_init_mode": INFERENCE_LAMBDA_INIT_MODE,
        **model_params,
    })
    metrics.update(summarize_diagnostics(diagnostics))
    metrics["train_lambda_init"] = None if bidder.train_lambda_init is None else float(bidder.train_lambda_init)

    metrics_path = trainval_config.outputs_dir / f"{label}_metrics.json"
    metrics_path.write_text(json.dumps(metrics, indent=2))

    return {
        "label": label,
        "metrics": metrics,
        "diagnostics_path": diagnostics_path,
        "metrics_path": metrics_path,
        "model_path": model_path,
        "model_params": model_params,
    }


In [4]:
split_metadata = ensure_train_val_split(
    source_config=source_config,
    target_config=trainval_config,
    val_fraction=VAL_FRACTION,
    seed=trainval_config.random_seed,
)

trainval_train_stats_df = pd.read_csv(trainval_config.data_config["train"]["stats_path"])
trainval_train_campaigns_df = pd.read_csv(trainval_config.data_config["train"]["campaigns_path"])
trainval_val_campaigns_df = pd.read_csv(trainval_config.data_config["test"]["campaigns_path"])
trainval_val_stats_df = pd.read_csv(trainval_config.data_config["test"]["stats_path"])

train_test_train_stats_df = pd.read_csv(traintest_source_config.data_config["train"]["stats_path"])
train_test_train_campaigns_df = pd.read_csv(traintest_source_config.data_config["train"]["campaigns_path"])

summary_df = pd.DataFrame(
    [
        {
            "split": "trainval_train",
            "campaigns": int(trainval_train_campaigns_df["campaign_id"].nunique()),
            "stats_rows": int(len(trainval_train_stats_df)),
        },
        {
            "split": "trainval_val",
            "campaigns": int(trainval_val_campaigns_df["campaign_id"].nunique()),
            "stats_rows": int(len(trainval_val_stats_df)),
        },
        {
            "split": "train_test_train",
            "campaigns": int(train_test_train_campaigns_df["campaign_id"].nunique()),
            "stats_rows": int(len(train_test_train_stats_df)),
        },
    ]
)

display(pd.DataFrame([split_metadata]))
display(summary_df)


,source_experiment,target_experiment,seed,val_fraction,train_campaigns,val_campaigns,train_stats_rows,val_stats_rows
0,exp_1_rnd_42_n10,exp_tune_smooth_drlb_dqn_lambda_train,42,0.2,1027,257,1246481,286690


,split,campaigns,stats_rows
0,trainval_train,1027,1246481
1,trainval_val,257,286690
2,train_test_train,1284,1533171


In [5]:
baseline_trainval_run = run_drlb_candidate(
    train_stats_df=trainval_train_stats_df,
    train_campaigns_df=trainval_train_campaigns_df,
    eval_campaigns_path=trainval_config.data_config["test"]["campaigns_path"],
    eval_stats_path=trainval_config.data_config["test"]["stats_path"],
    label="baseline_manual_trainval",
    model_params=BASELINE_MODEL_PARAMS,
    max_train_steps=MAX_TRAIN_STEPS,
    verbose=VERBOSE,
)

display(pd.DataFrame([baseline_trainval_run["metrics"]]))
print("baseline train/val model:", baseline_trainval_run["model_path"])


,cpc_relative,rmse,clicks_sum,quickspend,skipped_campaigns,time_inference_sec,time_overall_sec,label,exp_type,inference_lambda_init_mode,dqn_gamma,dqn_lr,dqn_target_update_interval,reward_net_lr,train_steps,last_dqn_loss,last_reward_net_loss,dqn_loss_mean,dqn_loss_p95,reward_net_loss_mean,reward_net_loss_p95,reward_signal_mean,lambda_final,train_lambda_init
0,78.451557,1.155505,5691.370161,0.046693,0,13.829302,17.834099,baseline_manual_trainval,improved_hybrid_drlb_smooth_eval,train_derived,1.0,0.0001,100,0.001,49200,0.60182,74.465088,304.152621,1324.026971,718652.64465,197.851617,7.117721,0.000009,0.000007


baseline train/val model: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/exp_tune_smooth_drlb_dqn_lambda_train/best_models/baseline_manual_trainval.pt


In [6]:
def objective(trial):
    model_params = {
        "dqn_gamma": trial.suggest_float("dqn_gamma", 0.90, 1.0),
        "dqn_lr": trial.suggest_float("dqn_lr", 1e-5, 5e-3, log=True),
        "dqn_target_update_interval": trial.suggest_int("dqn_target_update_interval", 10, 300, step=10),
        "reward_net_lr": trial.suggest_float("reward_net_lr", 1e-5, 5e-2, log=True),
    }
    run = run_drlb_candidate(
        train_stats_df=trainval_train_stats_df,
        train_campaigns_df=trainval_train_campaigns_df,
        eval_campaigns_path=trainval_config.data_config["test"]["campaigns_path"],
        eval_stats_path=trainval_config.data_config["test"]["stats_path"],
        label=f"trial_{trial.number:03d}",
        model_params=model_params,
        max_train_steps=MAX_TRAIN_STEPS,
        verbose=VERBOSE,
    )
    metrics = run["metrics"]
    trial.set_user_attr("clicks_sum", metrics["clicks_sum"])
    trial.set_user_attr("cpc_relative", metrics["cpc_relative"])
    trial.set_user_attr("rmse", metrics["rmse"])
    trial.set_user_attr("quickspend", metrics["quickspend"])
    trial.set_user_attr("train_lambda_init", metrics["train_lambda_init"])
    trial.set_user_attr("dqn_loss_mean", metrics["dqn_loss_mean"])
    return metrics["clicks_sum"]


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=trainval_config.random_seed),
)
study.optimize(objective, n_trials=max(1, N_TRIALS), n_jobs=1, show_progress_bar=True)

trials_df = study.trials_dataframe()
trials_path = trainval_config.outputs_dir / "optuna_trials.csv"
trials_df.to_csv(trials_path, index=False)

best_params = study.best_trial.params
best_params_path = trainval_config.best_params_dir / "best_trainval_params.json"
best_params_path.write_text(json.dumps(best_params, indent=2))

display(trials_df)
print("best params:", json.dumps(best_params, indent=2))
print("trials csv:", trials_path)
print("best params json:", best_params_path)


[I 2026-03-30 13:46:07,619] A new study created in memory with name: no-name-d467e31f-f51e-4bc5-846e-a4933e3b2fa8
Best trial: 0. Best value: 5664.26:  10%|█         | 1/10 [01:59<17:52, 119.16s/it]

[I 2026-03-30 13:48:06,796] Trial 0 finished with value: 5664.258884465285 and parameters: {'dqn_gamma': 0.9374540118847363, 'dqn_lr': 0.0036808608148776113, 'dqn_target_update_interval': 220, 'reward_net_lr': 0.0016383993835282328}. Best is trial 0 with value: 5664.258884465285.


Best trial: 1. Best value: 5691.37:  20%|██        | 2/10 [03:56<15:45, 118.14s/it]

[I 2026-03-30 13:50:04,222] Trial 1 finished with value: 5691.3701606426785 and parameters: {'dqn_gamma': 0.9156018640442437, 'dqn_lr': 2.6364803038431647e-05, 'dqn_target_update_interval': 20, 'reward_net_lr': 0.015994091701280463}. Best is trial 1 with value: 5691.3701606426785.


Best trial: 1. Best value: 5691.37:  30%|███       | 3/10 [05:58<13:57, 119.66s/it]

...

[I 2026-03-30 14:04:22,154] Trial 8 finished with value: 5691.3701606426785 and parameters: {'dqn_gamma': 0.9065051592985279, 'dqn_lr': 0.003639264345367793, 'dqn_target_update_interval': 290, 'reward_net_lr': 0.009777718780862659}. Best is trial 1 with value: 5691.3701606426785.


Best trial: 1. Best value: 5691.37: 100%|██████████| 10/10 [20:33<00:00, 123.34s/it]

[I 2026-03-30 14:06:41,029] Trial 9 finished with value: 5691.3701606426785 and parameters: {'dqn_gamma': 0.9304613769173371, 'dqn_lr': 1.834907204905544e-05, 'dqn_target_update_interval': 210, 'reward_net_lr': 0.00042472797953697195}. Best is trial 1 with value: 5691.3701606426785.


,number,value,datetime_start,datetime_complete,duration,params_dqn_gamma,params_dqn_lr,params_dqn_target_update_interval,params_reward_net_lr,user_attrs_clicks_sum,user_attrs_cpc_relative,user_attrs_dqn_loss_mean,user_attrs_quickspend,user_attrs_rmse,user_attrs_train_lambda_init,state
0,0,5664.258884,2026-03-30 13:46:07.639609,2026-03-30 13:48:06.795947,0 days 00:01:59.156338,0.937454,0.003681,220,0.001638,5664.258884,78.446443,5.920886,0.046693,1.155642,0.000007,COMPLETE
1,1,5691.370161,2026-03-30 13:48:06.797670,2026-03-30 13:50:04.221884,0 days 00:01:57.424214,0.915602,0.000026,20,0.015994,5691.370161,78.451557,438.430989,0.046693,1.155505,0.000007,COMPLETE
2,2,5602.310262,2026-03-30 13:50:04.223983,2026-03-30 13:52:05.689060,0 days 00:02:01.465077,0.960112,0.000815,10,0.038696,5602.310262,78.434225,15.924717,0.046693,1.156371,0.000007,COMPLETE
3,3,5691.370161,2026-03-30 13:52:05.690760,2026-03-30 13:54:01.826803,0 days 00:01:56.136043,0.983244,0.000037,60,0.000048,5691.370161,78.451557,521.965492,0.046693,1.155505,0.000007,COMPLETE
4,4,5691.370161,2026-03-30 13:54:01.828412,2026-03-30 13:55:57.395681,0 days 00:01:55.567269,0.930424,0.000261,130,0.000119,5691.370161,78.451557,90.985425,0.046693,1.155505,0.000007,COMPLETE
5,5,5691.370161,2026-03-30 13:55:57.397351,2026-03-30 13:57:59.720099,0 days 00:02:02.322748,0.961185,0.000024,90,0.000227,5691.370161,78.451557,414.056594,0.046693,1.155505,0.000007,COMPLETE
6,6,5581.044080,2026-03-30 13:57:59.722138,2026-03-30 13:59:59.696814,0 days 00:01:59.974676,0.945607,0.001316,60,0.000798,5581.044080,78.431135,11.179124,0.046693,1.156493,0.000007,COMPLETE
7,7,5691.370161,2026-03-30 13:59:59.698464,2026-03-30 14:02:05.810530,0 days 00:02:06.112066,0.959241,0.000013,190,0.000043,5691.370161,78.451557,263.942429,0.046693,1.155505,0.000007,COMPLETE
8,8,5691.370161,2026-03-30 14:02:05.812514,2026-03-30 14:04:22.154331,0 days 00:02:16.341817,0.906505,0.003639,290,0.009778,5691.370161,78.451557,6.319297,0.046693,1.155505,0.000007,COMPLETE
9,9,5691.370161,2026-03-30 14:04:22.156079,2026-03-30 14:06:41.029755,0 days 00:02:18.873676,0.930461,0.000018,210,0.000425,5691.370161,78.451557,335.810140,0.046693,1.155505,0.000007,COMPLETE


best params: {
  "dqn_gamma": 0.9156018640442437,
  "dqn_lr": 2.6364803038431647e-05,
  "dqn_target_update_interval": 20,
  "reward_net_lr": 0.015994091701280463
}
trials csv: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/exp_tune_smooth_drlb_dqn_lambda_train/outputs/optuna_trials.csv
best params json: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/exp_tune_smooth_drlb_dqn_lambda_train/best_params/best_trainval_params.json


In [7]:
best_trainval_run = run_drlb_candidate(
    train_stats_df=trainval_train_stats_df,
    train_campaigns_df=trainval_train_campaigns_df,
    eval_campaigns_path=trainval_config.data_config["test"]["campaigns_path"],
    eval_stats_path=trainval_config.data_config["test"]["stats_path"],
    label="best_trainval",
    model_params=best_params,
    max_train_steps=MAX_TRAIN_STEPS,
    verbose=VERBOSE,
)

baseline_train_test_run = run_drlb_candidate(
    train_stats_df=train_test_train_stats_df,
    train_campaigns_df=train_test_train_campaigns_df,
    eval_campaigns_path=traintest_source_config.data_config["test"]["campaigns_path"],
    eval_stats_path=traintest_source_config.data_config["test"]["stats_path"],
    label="baseline_train_test",
    model_params=BASELINE_MODEL_PARAMS,
    max_train_steps=MAX_TRAIN_STEPS,
    verbose=VERBOSE,
)

tuned_train_test_run = run_drlb_candidate(
    train_stats_df=train_test_train_stats_df,
    train_campaigns_df=train_test_train_campaigns_df,
    eval_campaigns_path=traintest_source_config.data_config["test"]["campaigns_path"],
    eval_stats_path=traintest_source_config.data_config["test"]["stats_path"],
    label="best_tuned_train_test",
    model_params=best_params,
    max_train_steps=MAX_TRAIN_STEPS,
    verbose=VERBOSE,
)

comparison_df = pd.DataFrame(
    [
        baseline_trainval_run["metrics"],
        best_trainval_run["metrics"],
        baseline_train_test_run["metrics"],
        tuned_train_test_run["metrics"],
    ]
)
comparison_path = trainval_config.outputs_dir / "comparison_metrics.csv"
comparison_df.to_csv(comparison_path, index=False)

display(comparison_df)
print("comparison csv:", comparison_path)


,cpc_relative,rmse,clicks_sum,quickspend,skipped_campaigns,time_inference_sec,time_overall_sec,label,exp_type,inference_lambda_init_mode,dqn_gamma,dqn_lr,dqn_target_update_interval,reward_net_lr,train_steps,last_dqn_loss,last_reward_net_loss,dqn_loss_mean,dqn_loss_p95,reward_net_loss_mean,reward_net_loss_p95,reward_signal_mean,lambda_final,train_lambda_init
0,78.451557,1.155505,5691.370161,0.046693,0,13.829302,17.834099,baseline_manual_trainval,improved_hybrid_drlb_smooth_eval,train_derived,1.000000,0.000100,100,0.001000,49200,0.601820,74.465088,304.152621,1324.026971,7.186526e+05,197.851617,7.117721,0.000009,0.000007
1,78.451557,1.155505,5691.370161,0.046693,0,14.915593,19.083641,best_trainval,improved_hybrid_drlb_smooth_eval,train_derived,0.915602,0.000026,20,0.015994,49200,271.064301,12.684307,438.430989,953.297867,4.151402e+06,173.802384,8.400647,0.000008,0.000007
2,109.784783,1.166326,31695.690642,0.025681,0,81.653783,104.367910,baseline_train_test,improved_hybrid_drlb_smooth_eval,train_derived,1.000000,0.000100,100,0.001000,60718,0.995476,71.576393,222.483058,1058.622833,5.255124e+05,304.655487,4.006017,0.000011,0.000007
3,109.778500,1.166652,31690.060317,0.025681,0,81.015675,103.447378,best_tuned_train_test,improved_hybrid_drlb_smooth_eval,train_derived,0.915602,0.000026,20,0.015994,60718,699.108765,23.396673,432.762426,960.067932,4.873182e+06,184.407440,16.274246,0.000006,0.000007


comparison csv: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/exp_tune_smooth_drlb_dqn_lambda_train/outputs/comparison_metrics.csv


In [8]:
summary = {
    "split_metadata": split_metadata,
    "inference_lambda_init_mode": INFERENCE_LAMBDA_INIT_MODE,
    "n_trials": int(N_TRIALS),
    "baseline_manual_trainval": baseline_trainval_run["metrics"],
    "best_trial_params": best_params,
    "best_trainval": best_trainval_run["metrics"],
    "baseline_train_test": baseline_train_test_run["metrics"],
    "best_tuned_train_test": tuned_train_test_run["metrics"],
    "study_best_value": float(study.best_trial.value),
}
summary_path = trainval_config.outputs_dir / "run_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print(json.dumps(summary, indent=2))
print("summary path:", summary_path)


{
  "split_metadata": {
    "source_experiment": "exp_1_rnd_42_n10",
    "target_experiment": "exp_tune_smooth_drlb_dqn_lambda_train",
    "seed": 42,
    "val_fraction": 0.2,
    "train_campaigns": 1027,
    "val_campaigns": 257,
    "train_stats_rows": 1246481,
    "val_stats_rows": 286690
  },
  "inference_lambda_init_mode": "train_derived",
  "n_trials": 10,
  "baseline_manual_trainval": {
    "cpc_relative": 78.45155686671056,
    "rmse": 1.1555050791507857,
    "clicks_sum": 5691.3701606426785,
    "quickspend": 0.04669260700389105,
    "skipped_campaigns": 0,
    "time_inference_sec": 13.829302072525024,
    "time_overall_sec": 17.834099054336548,
    "label": "baseline_manual_trainval",
    "exp_type": "improved_hybrid_drlb_smooth_eval",
    "inference_lambda_init_mode": "train_derived",
    "dqn_gamma": 1.0,
    "dqn_lr": 0.0001,
    "dqn_target_update_interval": 100,
    "reward_net_lr": 0.001,
    "train_steps": 49200,
    "last_dqn_loss": 0.6018204689025879,
    "last_rewar